Imports

In [12]:
import feedparser
import pandas as pd
from datetime import datetime, timedelta
import time
import sys
import requests
from bs4 import BeautifulSoup
import finviz
import os


Fraser Data Collection

In [ ]:
def scrape_fraser_timeline(url):
    """Scrapes date, title, description, and link info from the timeline."""
    try:
        # 1. Download the HTML content
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        # 2. Parse the HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # 3. Find the main container element
        # Your target container is class="timeline-events clusterize-scroll" and id="list-container"
        timeline_container = soup.find('div', id='list-container')
        
        if not timeline_container:
            print("Error: Main timeline container not found.")
            return []

        # 4. Find all individual event rows
        # The articles are inside <div class="row event-row active">
        event_rows = timeline_container.find_all('div', class_='event-row')

        data = []
        for row in event_rows:
            # 5. Extract data points using the specific classes
            
            # Date and Source/Title: <h2 class="list-item">
            header_element = row.find('h2', class_='list-item')
            header_text = header_element.text.strip() if header_element else 'N/A'
            
            # Description/Summary: <p class="list-item">
            summary_element = row.find('p', class_='list-item')
            summary = summary_element.text.strip() if summary_element else 'N/A'
            
            # Associated Link: <ul><li><a href="..." class="list-item">
            link_element = row.find('a', class_='list-item')
            
            link_title = link_element.text.strip() if link_element else 'N/A'
            link_url = link_element['href'] if link_element else 'N/A'
            
            # Prepend the base URL if the link is relative
            if link_url != 'N/A' and link_url.startswith('/'):
                link_url = 'https://fraser.stlouisfed.org' + link_url
            
            # Split the header into Date and Source
            if '|' in header_text:
                date, source = [part.strip() for part in header_text.split('|', 1)]
            else:
                date = header_text
                source = 'N/A'

            data.append({
                'Date': date,
                'Source': source,
                'Summary': summary,
                'Associated Link Title': link_title,
                'Associated Link URL': link_url
            })

        return data

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the request: {e}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return []
    
# Get Timeline Data from Fraser
# --- Execution ---
FINANCIAL_CRISIS_URL = "https://fraser.stlouisfed.org/timeline/financial-crisis"
fraser_financial_crisis_timeline_data = scrape_fraser_timeline(FINANCIAL_CRISIS_URL)
fraser_financial_crisis_timeline_df = pd.DataFrame(fraser_financial_crisis_timeline_data)
URL_COVID = "https://fraser.stlouisfed.org/timeline/covid-19-pandemic"
fraser_covid_timeline_data = scrape_fraser_timeline(URL_COVID)
fraser_covid_timeline_df = pd.DataFrame(fraser_covid_timeline_data)

# Save DataFrames to CSV files7
fraser_financial_crisis_timeline_df.to_csv('Data/fraser_financial_crisis_timeline.csv', index=False)    
fraser_covid_timeline_df.to_csv('Data/fraser_covid_timeline.csv', index=False)

Finviz Data Collection

In [4]:
# Defined list of Global Systemically Important Banks (G-SIBs) 
# and KBW Index members that work with Finviz
bank_tickers = [
    'JPM', 'BAC', 'WFC', 'C', 'GS', 'MS',  # US Giants
    'HSBC', 'SAN', 'BCS', 'DB', 'UBS',    # European Leaders (ADRs)
    'RY', 'TD'                             # Canadian Leaders
]

def fetch_bank_news(ticker_list):
    compiled_news = {}
    
    for ticker in ticker_list:
        try:
            print(f"Fetching news for {ticker}...")
            # Using the working individual stock news function
            news = finviz.get_news(ticker)
            
            # Store the latest 3 headlines
            compiled_news[ticker] = news[:3]
            
            # Small sleep to avoid rate limiting
            time.sleep(0.5) 
        except Exception as e:
            print(f"Could not fetch {ticker}: {e}")
            
    return compiled_news

# Run the fetcher
finviz_news = fetch_bank_news(bank_tickers)
print(finviz_news)
finviz_news_df = pd.DataFrame(finviz_news)
finviz_news_df.to_csv('Data/finviz_news.csv', index=False)

Fetching news for JPM...
Fetching news for BAC...
Fetching news for WFC...
Fetching news for C...
Fetching news for GS...
Fetching news for MS...
Fetching news for HSBC...
Fetching news for SAN...
Fetching news for BCS...
Fetching news for DB...
Fetching news for UBS...
Fetching news for RY...
Fetching news for TD...
{'JPM': [('2026-03-24 14:56', "JPMorgan Chase CEO Dimon: 'Deeply frustrated' with policies in America", 'https://www.youtube.com/watch?v=LAcX0bEl5ug', 'CNBC TV'), ('2026-03-24 14:56', "Jamie Dimon: 'Deeply frustrated' with policies in America that 'set us back'", 'https://www.youtube.com/shorts/7DGS8nHMl4A', 'CNBC TV'), ('2026-03-24 14:40', 'Fear Of Stablecoin Reward Limits Sends Circles Stock Down 20%', 'https://finance.yahoo.com/m/39b5b3ed-1495-3d5c-9706-7740d2168c28/fear-of-stablecoin-reward.html', 'CryptoProwl')], 'BAC': [('2026-03-24 17:48', 'Nebius Is Quietly Building Momentum In AI', 'https://finance.yahoo.com/sectors/technology/articles/nebius-quietly-building-mome

FINNHUB Data Collection

In [15]:
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv('FINNHUB_API_KEY')

# Your clearly defined list
institutions = ['JPM', 'GS', 'MS', 'HSBC', 'UBS', 'BLK', 'BK', 'SAN']

def fetch_all_institutional_news(ticker_list):
    all_news_items = []
    to_date = datetime.now().strftime('%Y-%m-%d')
    from_date = (datetime.now() - timedelta(days=14)).strftime('%Y-%m-%d')

    for ticker in ticker_list:
        print(f"Fetching news for {ticker}...")
        url = f'https://finnhub.io/api/v1/company-news?symbol={ticker}&from={from_date}&to={to_date}&token={API_KEY}'
        
        response = requests.get(url)
        if response.status_code == 200:
            news_data = response.json()
            
            # Add the Ticker symbol to each news entry so the CSV is organized
            for item in news_data:
                item['ticker_symbol'] = ticker # Custom column
                all_news_items.append(item)
        
        # Respect the free tier limit (60 calls/min)
        time.sleep(1) 
        
    return all_news_items

# 1. Run the fetcher
all_news = fetch_all_institutional_news(institutions)

# 2. Convert to DataFrame
df = pd.DataFrame(all_news)

# 3. Clean up the date (Finnhub uses Unix timestamps)
if not df.empty:
    df['date'] = pd.to_datetime(df['datetime'], unit='s')
    
    # 4. Save to CSV
    df.to_csv('Data/finnhub_news.csv', index=False)
    print(f"Successfully saved {len(df)} news items to Data/finnhub_news.csv")
else:
    print("No news found to save.")

Fetching news for JPM...
Fetching news for GS...
Fetching news for MS...
Fetching news for HSBC...
Fetching news for UBS...
Fetching news for BLK...
Fetching news for BK...
Fetching news for SAN...
Successfully saved 811 news items to Data/finnhub_news.csv
